# Bước 2: Thống kê mô tả và kiểm định

Gọi 6 hàm trong `dashboard/thong_ke.py` (hợp đồng: `docs/chi_so_thong_ke.md`, mục 1.1 đến 1.6) trên
`data/clean/vnu_that_sach.csv`, **chỉ dòng khảo sát thật** (`Nguon = Khao sat that`).

Chạy **hai lần**: (1) tất cả dòng thật; (2) bỏ các dòng trả lời một mức ở cả 9 câu (`Tra_loi_deu = True`).
Cuối notebook xuất `data/clean/ket_qua_thong_ke.csv`.

Quy tắc: GPA chỉ có 5 mức thứ bậc, nên dùng tần suất, tỉ lệ, trung vị, Spearman và kiểm định phi tham số.

In [1]:
   %pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

import pandas as pd

# Tự tìm thư mục gốc dự án (chạy từ notebooks/ hoặc từ thư mục gốc đều được)
GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dashboard" / "thong_ke.py").exists())
sys.path.insert(0, str(GOC / "dashboard"))
import thong_ke as tk

pd.set_option("display.width", 200, "display.max_columns", 30)

df_that = tk.doc_du_lieu(GOC / "data" / "clean" / "vnu_that_sach.csv")     # chỉ dòng thật
df_deu = df_that[~df_that[tk.COT_TRA_LOI_DEU].astype(bool)]                  # bỏ dòng trả lời một mức
print(f"Dòng thật: {len(df_that)} | trong đó bỏ dòng trả lời một mức còn: {len(df_deu)} "
      f"(loại {len(df_that) - len(df_deu)})")

Dòng thật: 1944 | trong đó bỏ dòng trả lời một mức còn: 1597 (loại 347)


In [3]:
def chay_tat_ca(df, ten_mau):
    """In toàn bộ chỉ số 1.1 đến 1.6 cho một tập mẫu."""
    print("=" * 78)
    print(f"{ten_mau}  (n = {len(df)})")
    print("=" * 78)

    print("\n1.1 Tần suất và tỉ lệ từng mức GPA")
    print(tk.tan_suat_ty_le_gpa(df).to_string(index=False))

    tl, so, n = tk.ty_le_gpa_muc_4_5(df)
    print(f"\n1.2 Tỉ lệ GPA mức 4-5 (từ 3.2 trở lên): {tl:.2f}% ({so}/{n})")

    muc, nhan = tk.gpa_trung_vi(df)
    print(f"\n1.3 Mức GPA trung vị: mức {muc} = {nhan}")

    print("\n1.4 Điểm trung bình các câu Likert (thang 1-5)")
    print(tk.diem_tb_likert(df).to_string(index=False))

    print("\n1.5 Tương quan hạng Spearman giữa GPA và các biến thời gian")
    print(tk.tuong_quan_spearman(df).to_string(index=False))

    print("\n1.6 So sánh GPA theo nhóm (kèm n)")
    for nhom, ten_kd in ([(g, "Mann-Whitney U") for g in tk.NHOM_MANN_WHITNEY]
                         + [(g, "Kruskal-Wallis H") for g in tk.NHOM_KRUSKAL]):
        bang, kd = tk.so_sanh_nhom(df, nhom, "GPA")
        p = kd["p-value"]
        p_txt = "không kiểm định được" if pd.isna(p) else ("p < 0.001" if p < 0.001 else f"p = {p:.3f}")
        print(f"\n- Theo {nhom}  [{ten_kd}, {p_txt}]" + (f"  ({kd['Ghi chú']})" if kd["Ghi chú"] else ""))
        print(bang.to_string(index=False))

## Lần 1: tất cả dòng khảo sát thật

In [4]:
chay_tat_ca(df_that, "LẦN 1 - Tất cả dòng thật")

LẦN 1 - Tất cả dòng thật  (n = 1944)

1.1 Tần suất và tỉ lệ từng mức GPA
 Mức                   Nhãn  Số SV  Tỉ lệ (%)
   1         Yếu (Dưới 2.0)     59       3.03
   2 Trung bình (2.0 - 2.5)    106       5.45
   3        Khá (2.5 - 3.2)   1079      55.50
   4       Giỏi (3.2 - 3.6)    607      31.22
   5    Xuất sắc (Trên 3.6)     93       4.78

1.2 Tỉ lệ GPA mức 4-5 (từ 3.2 trở lên): 36.01% (700/1944)

1.3 Mức GPA trung vị: mức 3 = Khá (2.5 - 3.2)

1.4 Điểm trung bình các câu Likert (thang 1-5)
                 Yếu tố  Điểm TB (1-5)    n
  Chất lượng Giảng viên           4.33 1944
Sự hỗ trợ từ Giảng viên           4.18 1944
   Chương trình đào tạo           4.12 1944
         Cơ sở vật chất           4.07 1944
    Sự hỗ trợ từ Trường           3.99 1944
Sự cạnh tranh trong lớp           3.94 1944
    Ảnh hưởng từ bạn bè           3.83 1944
    Phương pháp học tập           3.62 1944
   Mức độ thích nghi ĐH           3.46 1944

1.5 Tương quan hạng Spearman giữa GPA và các biến thời g

## Lần 2: bỏ các dòng trả lời một mức ở cả 9 câu

In [5]:
chay_tat_ca(df_deu, "LẦN 2 - Bỏ dòng trả lời một mức")

LẦN 2 - Bỏ dòng trả lời một mức  (n = 1597)

1.1 Tần suất và tỉ lệ từng mức GPA
 Mức                   Nhãn  Số SV  Tỉ lệ (%)
   1         Yếu (Dưới 2.0)     43       2.69
   2 Trung bình (2.0 - 2.5)     88       5.51
   3        Khá (2.5 - 3.2)    885      55.42
   4       Giỏi (3.2 - 3.6)    502      31.43
   5    Xuất sắc (Trên 3.6)     79       4.95

1.2 Tỉ lệ GPA mức 4-5 (từ 3.2 trở lên): 36.38% (581/1597)

1.3 Mức GPA trung vị: mức 3 = Khá (2.5 - 3.2)

1.4 Điểm trung bình các câu Likert (thang 1-5)
                 Yếu tố  Điểm TB (1-5)    n
  Chất lượng Giảng viên           4.38 1597
Sự hỗ trợ từ Giảng viên           4.20 1597
   Chương trình đào tạo           4.12 1597
         Cơ sở vật chất           4.06 1597
    Sự hỗ trợ từ Trường           3.97 1597
Sự cạnh tranh trong lớp           3.90 1597
    Ảnh hưởng từ bạn bè           3.77 1597
    Phương pháp học tập           3.52 1597
   Mức độ thích nghi ĐH           3.32 1597

1.5 Tương quan hạng Spearman giữa GPA và các biến

## So sánh độ nhạy giữa hai lần chạy

Nếu số liệu gần như không đổi thì việc có hay không có các dòng trả lời một mức không làm đổi kết luận.
Tỉ lệ GPA mức 4-5, trung vị và các hệ số Spearman gần như không đổi. Điểm trung bình Likert dịch chuyển tối đa
khoảng 0,14 điểm (có mục tăng, có mục giảm); hai mục thấp nhất vẫn là thích nghi ĐH và phương pháp học tập.

In [6]:
def tom_tat(df):
    tl, _, _ = tk.ty_le_gpa_muc_4_5(df)
    out = {"n": len(df), "Tỉ lệ GPA mức 4-5 (%)": round(tl, 2), "Mức GPA trung vị": tk.gpa_trung_vi(df)[0]}
    for _, r in tk.diem_tb_likert(df).iterrows():
        out[f"TB {r['Yếu tố']}"] = r["Điểm TB (1-5)"]
    for _, r in tk.tuong_quan_spearman(df).iterrows():
        out[f"Spearman {r['Biến']}"] = r["Hệ số Spearman"]
    return pd.Series(out)

do_nhay = pd.DataFrame({"Tất cả dòng thật": tom_tat(df_that), "Bỏ dòng trả lời một mức": tom_tat(df_deu)})
do_nhay["Chênh lệch"] = (do_nhay.iloc[:, 1] - do_nhay.iloc[:, 0]).round(3)
print(do_nhay.to_string())

                              Tất cả dòng thật  Bỏ dòng trả lời một mức  Chênh lệch
n                                     1944.000                 1597.000    -347.000
Tỉ lệ GPA mức 4-5 (%)                   36.010                   36.380       0.370
Mức GPA trung vị                         3.000                    3.000       0.000
TB Chất lượng Giảng viên                 4.330                    4.380       0.050
TB Sự hỗ trợ từ Giảng viên               4.180                    4.200       0.020
TB Chương trình đào tạo                  4.120                    4.120       0.000
TB Cơ sở vật chất                        4.070                    4.060      -0.010
TB Sự hỗ trợ từ Trường                   3.990                    3.970      -0.020
TB Sự cạnh tranh trong lớp               3.940                    3.900      -0.040
TB Ảnh hưởng từ bạn bè                   3.830                    3.770      -0.060
TB Phương pháp học tập                   3.620                    3.520     

## Xuất `data/clean/ket_qua_thong_ke.csv`

Mỗi dòng một con số: `chi_so, nhom, n, gia_tri, p` (kèm `mau` = tập mẫu và `ghi_chu`).
Với chỉ số theo nhóm, `p` là p-value của cả phép so sánh nên lặp lại ở mọi nhóm của cùng chỉ số.
Nhóm có n < 30 được giữ lại nhưng ghi chú "không kết luận".

In [7]:
kq = pd.concat([
    tk.tao_bang_ket_qua(df_that, "Tất cả dòng thật"),
    tk.tao_bang_ket_qua(df_deu, "Bỏ dòng trả lời một mức"),
], ignore_index=True)

# Kiểm tra: đủ mọi nhóm của mọi biến phân nhóm, ở cả hai tập mẫu
for mau, d in [("Tất cả dòng thật", df_that), ("Bỏ dòng trả lời một mức", df_deu)]:
    for nhom in tk.NHOM_MANN_WHITNEY + tk.NHOM_KRUSKAL:
        co = set(kq[(kq["mau"] == mau) & (kq["chi_so"] == f"Tỉ lệ GPA mức 4-5 (%) theo {nhom}")]["nhom"])
        can = set(d[nhom].dropna().unique())
        assert co == can, f"Thiếu nhóm ở {nhom} / {mau}: {can - co}"

out = GOC / "data" / "clean" / "ket_qua_thong_ke.csv"
kq.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Đã ghi {out.name}: {len(kq)} dòng, các cột: {list(kq.columns)}")
print(kq["mau"].value_counts().to_string())
kq.head(12)

Đã ghi ket_qua_thong_ke.csv: 664 dòng, các cột: ['chi_so', 'nhom', 'n', 'gia_tri', 'p', 'mau', 'ghi_chu']
mau
Tất cả dòng thật           332
Bỏ dòng trả lời một mức    332


,chi_so,nhom,n,gia_tri,p,mau,ghi_chu
0,Tần suất GPA,Yếu (Dưới 2.0),1944,59.0000,NaN,Tất cả dòng thật,
1,Tỉ lệ GPA (%),Yếu (Dưới 2.0),1944,3.0300,NaN,Tất cả dòng thật,
2,Tần suất GPA,Trung bình (2.0 - 2.5),1944,106.0000,NaN,Tất cả dòng thật,
3,Tỉ lệ GPA (%),Trung bình (2.0 - 2.5),1944,5.4500,NaN,Tất cả dòng thật,
4,Tần suất GPA,Khá (2.5 - 3.2),1944,1079.0000,NaN,Tất cả dòng thật,
5,Tỉ lệ GPA (%),Khá (2.5 - 3.2),1944,55.5000,NaN,Tất cả dòng thật,
6,Tần suất GPA,Giỏi (3.2 - 3.6),1944,607.0000,NaN,Tất cả dòng thật,
7,Tỉ lệ GPA (%),Giỏi (3.2 - 3.6),1944,31.2200,NaN,Tất cả dòng thật,
8,Tần suất GPA,Xuất sắc (Trên 3.6),1944,93.0000,NaN,Tất cả dòng thật,
9,Tỉ lệ GPA (%),Xuất sắc (Trên 3.6),1944,4.7800,NaN,Tất cả dòng thật,
